# Deep Learning Fundamentals: Convolutional Neural Networks (CNNs)

In this notebook, we will explore **Convolutional Neural Networks (CNNs)**, which are specialized neural networks for processing grid-like data such as images.

## Learning objectives

By the end of this notebook, you will be able to:
- Understand the key components of CNNs: convolution, pooling, and fully connected layers
- Load and preprocess image data using PyTorch and torchvision
- Build a CNN architecture for image classification
- Visualize convolutional kernels/filters
- Visualize feature maps at different layers
- Train a CNN on the CIFAR-10 dataset
- Evaluate CNN performance on image classification tasks

## Setup: Import libraries

We'll use PyTorch for building CNNs, torchvision for image datasets, and matplotlib for visualization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset
import os

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Imports ready.')
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {device}')

## 1. Load and Explore the CIFAR-10 Dataset

CIFAR-10 is a dataset of 60,000 32x32 color images in 10 classes:
- airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck

We'll try to download it from the internet, with a fallback to local data if needed.

In [ ]:
# Define transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize to [-1, 1]
])

# Try to load CIFAR-10 dataset
try:
    # Try downloading from internet
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                            download=True, transform=transform)
    testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                           download=True, transform=transform)
    print('CIFAR-10 dataset loaded successfully from download.')
except Exception as e:
    print(f'Could not download CIFAR-10: {e}')
    print('Attempting to load from local data directory...')
    
    # Fallback: Try to load from local data directory
    data_dir = '../data/cifar-10-batches-py'
    if os.path.exists(data_dir):
        trainset = torchvision.datasets.CIFAR10(root='../data', train=True,
                                                download=False, transform=transform)
        testset = torchvision.datasets.CIFAR10(root='../data', train=False,
                                               download=False, transform=transform)
        print('CIFAR-10 dataset loaded from local directory.')
    else:
        # Last resort: Create a small synthetic dataset for demonstration
        print('Creating synthetic dataset for demonstration...')
        # Create synthetic data: 1000 samples, 3 channels, 32x32 pixels
        train_images = torch.randn(1000, 3, 32, 32)
        train_labels = torch.randint(0, 10, (1000,))
        test_images = torch.randn(200, 3, 32, 32)
        test_labels = torch.randint(0, 10, (200,))
        
        trainset = TensorDataset(train_images, train_labels)
        testset = TensorDataset(test_images, test_labels)
        print('Synthetic dataset created.')

# Create data loaders
batch_size = 64
trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=0)
testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=0)

# CIFAR-10 class names
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

print(f'\nTraining set size: {len(trainset)}')
print(f'Test set size: {len(testset)}')
print(f'Batch size: {batch_size}')
print(f'Number of batches (train): {len(trainloader)}')

### Visualize sample images

Let's look at some examples from the dataset to understand what we're working with.

In [ ]:
# Function to unnormalize and display images
def imshow(img):
    img = img / 2 + 0.5  # Unnormalize from [-1, 1] to [0, 1]
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.axis('off')

# Get a batch of training images
dataiter = iter(trainloader)
images, labels = next(dataiter)

# Show a grid of images
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    if i < len(images):
        img = images[i]
        img = img / 2 + 0.5  # Unnormalize
        npimg = img.numpy()
        ax.imshow(np.transpose(npimg, (1, 2, 0)))
        ax.set_title(classes[labels[i]])
        ax.axis('off')

plt.tight_layout()
plt.suptitle('Sample Images from CIFAR-10', y=1.02, fontsize=16)
plt.show()

print(f'\nImage shape: {images[0].shape}')  # [channels, height, width]
print(f'Label: {labels[0].item()} ({classes[labels[0]]})')

## 2. Build a Convolutional Neural Network

We'll create a CNN with:
- **Convolutional layers**: Extract features from images using learnable filters
- **Pooling layers**: Reduce spatial dimensions and computational cost
- **Fully connected layers**: Make final classification decisions

### CNN Architecture:
1. Conv Layer 1: 3 input channels → 32 output channels, 3×3 kernel
2. ReLU activation
3. Max Pooling: 2×2
4. Conv Layer 2: 32 → 64 channels, 3×3 kernel
5. ReLU activation
6. Max Pooling: 2×2
7. Conv Layer 3: 64 → 64 channels, 3×3 kernel
8. ReLU activation
9. Flatten
10. Fully Connected Layer 1: 64×6×6 → 64
11. ReLU activation
12. Fully Connected Layer 2: 64 → 10 (classes)

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Convolutional layers
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1)
        
        # Pooling layer
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Fully connected layers
        # After 2 pooling layers: 32x32 -> 16x16 -> 8x8, then conv3 processes at 8x8
        self.fc1 = nn.Linear(64 * 8 * 8, 64)
        self.fc2 = nn.Linear(64, 10)  # 10 classes
        
        # Dropout for regularization
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        # Convolutional layers with ReLU and pooling
        x = self.pool(F.relu(self.conv1(x)))  # 32x32 -> 16x16
        x = self.pool(F.relu(self.conv2(x)))  # 16x16 -> 8x8
        x = F.relu(self.conv3(x))             # 8x8 -> 8x8
        
        # Flatten
        x = x.view(-1, 64 * 8 * 8)
        
        # Fully connected layers
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

# Create the model
model = SimpleCNN().to(device)

# Print model architecture
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

### Visualize Convolutional Kernels/Filters

Let's visualize the filters in the first convolutional layer to see what patterns they learn.

In [ ]:
# Get the weights of the first convolutional layer
conv1_weights = model.conv1.weight.data.cpu()

# Visualize the first 32 filters
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i < conv1_weights.shape[0]:
        # Get the i-th filter (shape: [3, 3, 3] for RGB)
        kernel = conv1_weights[i]
        
        # Normalize to [0, 1] for visualization
        kernel = kernel - kernel.min()
        kernel = kernel / (kernel.max() + 1e-8)
        
        # Convert from [channels, height, width] to [height, width, channels] for display
        kernel_img = kernel.permute(1, 2, 0).numpy()
        
        ax.imshow(kernel_img)
        ax.set_title(f'Filter {i+1}')
        ax.axis('off')

plt.tight_layout()
plt.suptitle('Convolutional Filters (First Layer - Before Training)', y=1.0, fontsize=16)
plt.show()

print(f'\nFilter shape: {conv1_weights[0].shape}')  # [out_channels, in_channels, height, width]

## 3. Define Loss Function and Optimizer

For multi-class classification:
- **Loss function**: Cross-entropy loss
- **Optimizer**: Adam (adaptive learning rate)

In [ ]:
# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

print('Loss function: CrossEntropyLoss')
print('Optimizer: Adam')
print(f'Learning rate: 0.001')

## 4. Train the CNN

We'll train for a few epochs to demonstrate the process. In practice, you'd train for more epochs and monitor validation performance.

In [ ]:
# Training parameters
num_epochs = 5  # Short training for demonstration

# Storage for metrics
train_losses = []
train_accuracies = []
test_accuracies = []

print('Starting training...\n')

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for i, (inputs, labels) in enumerate(trainloader):
        # Move data to device
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Zero the parameter gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        # Print progress every 100 batches
        if (i + 1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(trainloader)}], '
                  f'Loss: {loss.item():.4f}')
    
    # Calculate epoch metrics
    epoch_loss = running_loss / len(trainloader)
    epoch_acc = 100 * correct / total
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_acc)
    
    # Evaluate on test set
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    test_acc = 100 * correct / total
    test_accuracies.append(test_acc)
    
    print(f'\nEpoch [{epoch+1}/{num_epochs}]')
    print(f'Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.2f}%')
    print(f'Test Acc: {test_acc:.2f}%\n')

print('Training finished!')

## 5. Visualize Training Progress

In [ ]:
# Plot training metrics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(range(1, num_epochs + 1), train_losses, 'b-o', label='Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss over Epochs')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(range(1, num_epochs + 1), train_accuracies, 'b-o', label='Train Accuracy')
ax2.plot(range(1, num_epochs + 1), test_accuracies, 'r-o', label='Test Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy over Epochs')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nFinal Training Accuracy: {train_accuracies[-1]:.2f}%')
print(f'Final Test Accuracy: {test_accuracies[-1]:.2f}%')

### Visualize Learned Filters

After training, let's see how the filters have changed and what features they've learned to detect.

In [ ]:
# Get the weights of the first convolutional layer after training
conv1_weights_trained = model.conv1.weight.data.cpu()

# Visualize the first 32 filters
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i < conv1_weights_trained.shape[0]:
        # Get the i-th filter
        kernel = conv1_weights_trained[i]
        
        # Normalize to [0, 1] for visualization
        kernel = kernel - kernel.min()
        kernel = kernel / (kernel.max() + 1e-8)
        
        # Convert to displayable format
        kernel_img = kernel.permute(1, 2, 0).numpy()
        
        ax.imshow(kernel_img)
        ax.set_title(f'Filter {i+1}')
        ax.axis('off')

plt.tight_layout()
plt.suptitle('Learned Convolutional Filters (After Training)', y=1.0, fontsize=16)
plt.show()

## 6. Visualize Feature Maps

Feature maps show the output of convolutional layers when an image is passed through the network. They reveal what features the network detects at different layers.

In [ ]:
# Get a single test image
dataiter = iter(testloader)
test_images, test_labels = next(dataiter)
test_image = test_images[0:1].to(device)  # Get first image and keep batch dimension

# Display the original image
plt.figure(figsize=(4, 4))
img_display = test_images[0] / 2 + 0.5  # Unnormalize
npimg = img_display.numpy()
plt.imshow(np.transpose(npimg, (1, 2, 0)))
plt.title(f'Original Image: {classes[test_labels[0]]}')
plt.axis('off')
plt.show()

# Function to get feature maps from specific layers
def get_feature_maps(model, x, layer_num):
    """
    Extract feature maps from a specific convolutional layer.
    layer_num: 1, 2, or 3 for conv1, conv2, or conv3
    """
    model.eval()
    with torch.no_grad():
        if layer_num == 1:
            x = F.relu(model.conv1(x))
        elif layer_num == 2:
            x = model.pool(F.relu(model.conv1(x)))
            x = F.relu(model.conv2(x))
        elif layer_num == 3:
            x = model.pool(F.relu(model.conv1(x)))
            x = model.pool(F.relu(model.conv2(x)))
            x = F.relu(model.conv3(x))
    return x

# Get feature maps from all three layers
feature_maps_1 = get_feature_maps(model, test_image, 1)
feature_maps_2 = get_feature_maps(model, test_image, 2)
feature_maps_3 = get_feature_maps(model, test_image, 3)

print(f'Feature maps shape after Conv1: {feature_maps_1.shape}')  # [1, 32, 32, 32]
print(f'Feature maps shape after Conv2: {feature_maps_2.shape}')  # [1, 64, 16, 16]
print(f'Feature maps shape after Conv3: {feature_maps_3.shape}')  # [1, 64, 8, 8]

In [ ]:
# Visualize feature maps from first layer
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
feature_maps = feature_maps_1[0].cpu()  # Remove batch dimension

for i, ax in enumerate(axes.flat):
    if i < feature_maps.shape[0]:
        ax.imshow(feature_maps[i], cmap='viridis')
        ax.set_title(f'Map {i+1}')
        ax.axis('off')

plt.tight_layout()
plt.suptitle('Feature Maps from First Convolutional Layer', y=1.0, fontsize=16)
plt.show()

In [ ]:
# Visualize feature maps from second layer
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
feature_maps = feature_maps_2[0].cpu()  # Remove batch dimension

for i, ax in enumerate(axes.flat):
    if i < min(32, feature_maps.shape[0]):  # Show first 32 maps
        ax.imshow(feature_maps[i], cmap='viridis')
        ax.set_title(f'Map {i+1}')
        ax.axis('off')

plt.tight_layout()
plt.suptitle('Feature Maps from Second Convolutional Layer', y=1.0, fontsize=16)
plt.show()

## 7. Evaluate Model Performance

Let's evaluate the model's performance on the test set and see predictions for individual images.

In [ ]:
# Evaluate on test set
model.eval()
correct = 0
total = 0
class_correct = [0] * 10
class_total = [0] * 10

with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        # Per-class accuracy
        c = (predicted == labels).squeeze()
        for i in range(len(labels)):
            label = labels[i]
            class_correct[label] += c[i].item()
            class_total[label] += 1

print(f'Overall Test Accuracy: {100 * correct / total:.2f}%\n')

print('Per-class Accuracy:')
for i in range(10):
    if class_total[i] > 0:
        acc = 100 * class_correct[i] / class_total[i]
        print(f'{classes[i]:>10s}: {acc:.2f}%')

In [ ]:
# Show some predictions
dataiter = iter(testloader)
images, labels = next(dataiter)
images_device = images.to(device)

# Get predictions
model.eval()
with torch.no_grad():
    outputs = model(images_device)
    _, predicted = torch.max(outputs, 1)
    probabilities = F.softmax(outputs, dim=1)

# Display images with predictions
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for i, ax in enumerate(axes.flat):
    if i < len(images):
        img = images[i] / 2 + 0.5  # Unnormalize
        npimg = img.numpy()
        ax.imshow(np.transpose(npimg, (1, 2, 0)))
        
        pred_label = classes[predicted[i]]
        true_label = classes[labels[i]]
        prob = probabilities[i][predicted[i]].item() * 100
        
        color = 'green' if predicted[i] == labels[i] else 'red'
        ax.set_title(f'True: {true_label}\nPred: {pred_label}\n({prob:.1f}%)', 
                     color=color, fontsize=9)
        ax.axis('off')

plt.tight_layout()
plt.suptitle('Model Predictions (Green=Correct, Red=Incorrect)', y=1.02, fontsize=16)
plt.show()

## 8. Exercises

Try these exercises to deepen your understanding:

1. **Modify the architecture**: 
   - Add more convolutional layers
   - Change the number of filters
   - Experiment with different kernel sizes

2. **Data augmentation**: 
   - Add random flips, rotations, or color jittering to the training data
   - See if it improves generalization

3. **Learning rate experiments**:
   - Try different learning rates (0.0001, 0.01)
   - Implement learning rate scheduling

4. **Batch normalization**:
   - Add batch normalization layers after convolutions
   - Compare training speed and final accuracy

5. **Visualization**:
   - Visualize feature maps from different layers for correctly vs incorrectly classified images
   - Create an interactive tool to explore what different filters detect

## Summary

In this notebook, we:

1. ✅ Loaded and explored the CIFAR-10 dataset
2. ✅ Built a CNN architecture with convolutional, pooling, and fully connected layers
3. ✅ Visualized convolutional kernels/filters before and after training
4. ✅ Trained the CNN using cross-entropy loss and Adam optimizer
5. ✅ Visualized feature maps at different layers to understand what the network learns
6. ✅ Evaluated model performance on the test set
7. ✅ Explored per-class accuracy and predictions

### Key Takeaways:

- **CNNs use convolutional layers** to automatically learn hierarchical features from images
- **Early layers** detect simple features (edges, colors), while **deeper layers** detect complex patterns
- **Pooling layers** reduce spatial dimensions and make the network more robust to small translations
- **Feature maps** provide insights into what the network has learned at each layer
- **Visualization** helps us understand and debug neural networks

### Next Steps:

- Explore transfer learning with pre-trained models (ResNet, VGG)
- Learn about more advanced architectures (Inception, EfficientNet)
- Apply CNNs to your own image classification problems
- Study attention mechanisms and vision transformers